# Agreement of LLM Judges in Persona Contradiction Detection

**Author:** Matyáš Martinek
**Course:** Probability and Statistics
**Language:** Python

## Objective

This project investigates whether different large language models agree when
judging contradictions between a complete persona and a dialogue.

Each LLM judge receives the same persona–dialogue example and predicts one of
two labels:

- `CONTRADICTION`
- `NO_CONTRADICTION`

The primary goal is to measure inter-judge agreement and determine whether
different LLM judges exhibit systematically different decision behaviour.

The project further investigates whether agreement is related to properties
of the input, including persona length, dialogue length, lexical overlap, and
the number of augmented persona facts.

## 1. Data source and provenance

The base data come from the **PersonaChat** dataset introduced by
Zhang et al. (2018). PersonaChat contains multi-turn English dialogues in
which each participant is assigned a persona represented by several natural
language statements.

For this project, I use the PersonaChat distribution available on Hugging Face:

https://huggingface.co/datasets/awsaf49/persona-chat

The original dataset is described in:

> Zhang, S., Dinan, E., Urbanek, J., Szlam, A., Kiela, D., & Weston, J.
> (2018). *Personalizing Dialogue Agents: I have a dog, do you have pets too?*
> Proceedings of ACL 2018, pp. 2204–2213.
> DOI: 10.18653/v1/P18-1205.

Original paper:

https://aclanthology.org/P18-1205/

### Derived data

The PersonaChat data were processed as part of a bachelor-thesis project on
persona consistency.

Persona facts were automatically grounded in their corresponding dialogues.
Grounding estimates whether a persona fact is supported by evidence in the
dialogue.

Contradiction-oriented persona variants were subsequently generated using
deterministic rule-based augmentations. Multiple supported facts belonging to
the same persona may be modified simultaneously while the corresponding
dialogue remains unchanged.

The resulting profile-level dataset is stored in:

`personachat_augmented_v2.parquet`

Each row represents one speaker persona and its dialogue and contains:

- the complete original persona,
- the complete augmented persona,
- the unchanged dialogue,
- the number and types of applied augmentations,
- grounding information for the changed facts,
- augmentation provenance.

For rule-based profiles, an augmented persona is assigned the expected relation
`contradiction` when at least one changed persona fact is grounded in the
dialogue.

These automatically derived relations are treated as **expected labels**, not
as manually verified ground truth.

In [19]:
from pathlib import Path
import re

import pandas as pd

from judge_config import (
    PROMPT_ID,
    build_judge_prompt,
    parse_judge_output,
)


RANDOM_SEED = 42

DATA_DIR = Path("data")
JUDGMENTS_DIR = DATA_DIR / "outputs"

SOURCE_DATA_PATH = (
    DATA_DIR
    / "personachat_augmented_v2.parquet"
)

source_df = pd.read_parquet(
    SOURCE_DATA_PATH
)

print(f"Rows: {len(source_df):,}")
print(
    f"Dialogues: "
    f"{source_df['dialogue_id'].nunique():,}"
)

print(
    "\nAugmentation families:"
)

print(
    source_df[
        "augmentation_family"
    ].value_counts()
)

source_df.head()

Rows: 26,368
Dialogues: 8,939

Augmentation families:
augmentation_family
persona_swap    17878
rule             8490
Name: count, dtype: int64


,augmentation_id,dialogue_id,speaker,augmentation_family,augmentation_type,augmentation_method,dialogue_text,original_persona,augmented_persona,persona_fact_count,...,contradiction_strengths,generation_notes,rule_versions,changed_expected_relations,changed_grounding_labels,changed_grounding_scores,changed_best_utterances,expected_relation,donor_dialogue_id,seed
0,aug_000000,0,speaker_1,rule,preference_negation,rule_based,"SELF: hi , how are you doing ? i am getting re...","[i like canning and whittling., to stay in sha...","[i do not like canning and whittling., to stay...",4,...,[strong],[Preference polarity was negated using a deter...,[v1],[contradiction],[grounded],[0.9796050786972046],[i am ! for my hobby i like to do canning or s...,contradiction,NaN,42
1,aug_000001,0,speaker_2,rule,preference_negation,rule_based,"PARTNER: hi , how are you doing ? i am getting...","[i like to remodel homes., i like to go huntin...","[i do not like to remodel homes., i do not lik...",4,...,"[strong, strong]",[Preference polarity was negated using a deter...,"[v1, v1]","[contradiction, contradiction]","[grounded, grounded]","[0.8077989220619202, 0.8201733231544495]",[i also remodel homes when i am not out bow hu...,contradiction,NaN,42
2,aug_000002,1,speaker_1,rule,preference_negation,rule_based,"SELF: hi , how are you doing today ?\nPARTNER:...","[i wish i could live forever., i only date peo...","[i wish i could live forever., i only date peo...",4,...,[strong],[Preference polarity was negated using a deter...,[v1],[contradiction],[grounded],[0.9774878025054932],"[i really enjoy free diving , how about you , ...",contradiction,NaN,42
3,aug_000003,1,speaker_2,rule,family_composition_swap,rule_based,"PARTNER: hi , how are you doing today ?\nSELF:...","[my mom is my best friend., i have four sister...","[my mom is my best friend., i have 2 sisters.,...",4,...,[medium_strong],[Family composition count was replaced with a ...,[v1],[contradiction],[grounded],[0.9484180808067322],[i am spending time with my 4 sisters what are...,contradiction,NaN,42
4,aug_000004,2,speaker_1,rule,preference_negation,rule_based,"SELF: we all live in a yellow submarine , a ye...","[i love the beatles., i have trouble getting a...","[i hate the beatles., i have trouble getting a...",4,...,[strong],[Preference polarity was negated using a deter...,[v1],[contradiction],[grounded],[0.8553723692893982],"[lol . i am shy , anything to break the ice , ...",contradiction,NaN,42


In [20]:
eligible_df = source_df[
    source_df["augmentation_family"].eq("rule")
    & source_df["expected_relation"].eq("contradiction")
    & source_df["num_grounded_augmented_facts"].gt(0)
].copy()


print(
    f"Eligible rule-augmented profiles: "
    f"{len(eligible_df):,}"
)

print(
    f"Unique dialogues: "
    f"{eligible_df['dialogue_id'].nunique():,}"
)

Eligible rule-augmented profiles: 8,469
Unique dialogues: 6,399


In [21]:
profile_summary = pd.DataFrame(
    {
        "num_augmented_facts": (
            eligible_df[
                "num_augmented_facts"
            ].describe()
        ),
        "num_grounded_augmented_facts": (
            eligible_df[
                "num_grounded_augmented_facts"
            ].describe()
        ),
    }
)

profile_summary

,num_augmented_facts,num_grounded_augmented_facts
count,8469.000000,8469.000000
mean,1.502775,1.499351
std,0.740965,0.739694
min,1.000000,1.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,2.000000,2.000000
max,5.000000,5.000000


## 2. Experimental dataset construction

The experiment is based on complete persona–dialogue profiles rather than
individual persona facts.

For every selected profile, two matched evaluation examples are constructed:

1. **Original variant** – the complete original persona is paired with its
   dialogue and is expected to be `NO_CONTRADICTION`.
2. **Augmented variant** – the complete augmented persona is paired with the
   same dialogue and is expected to be `CONTRADICTION`.

The dialogue is therefore identical within each pair. The only difference is
that one or more persona statements have been modified in the augmented
variant.

Only rule-based profiles with expected relation `contradiction` are used. Such
profiles contain at least one changed persona fact that was grounded in the
dialogue.

To reduce dependence between observations, at most one speaker profile is
selected from each dialogue. Sampling is deterministic using a fixed random
seed.

Persona-swap examples are excluded from the main experiment because they are
currently contradiction candidates rather than verified contradiction-oriented
profiles.

In [22]:
N_PAIRS = 1000

In [23]:
sampling_pool = (
    eligible_df
    .sample(
        frac=1,
        random_state=RANDOM_SEED,
    )
    .drop_duplicates(
        subset="dialogue_id"
    )
    .reset_index(drop=True)
)

if len(sampling_pool) < N_PAIRS:
    raise ValueError(
        f"Only {len(sampling_pool):,} unique dialogues "
        f"are available, but {N_PAIRS:,} pairs were requested."
    )

selected_pairs_df = (
    sampling_pool
    .head(N_PAIRS)
    .copy()
)

selected_pairs_df = selected_pairs_df.rename(
    columns={
        "num_augmented_facts": (
            "pair_num_augmented_facts"
        ),
        "num_grounded_augmented_facts": (
            "pair_num_grounded_augmented_facts"
        ),
    }
)

print(
    f"Selected profiles: "
    f"{len(selected_pairs_df):,}"
)

print(
    f"Unique dialogues: "
    f"{selected_pairs_df['dialogue_id'].nunique():,}"
)

Selected profiles: 1,000
Unique dialogues: 1,000


In [24]:
def format_persona(
    persona,
) -> str:
    """
    Format a list of persona statements for LLM evaluation.
    """
    return "\n".join(
        f"- {fact}"
        for fact in persona
    )

In [25]:
original_df = selected_pairs_df.copy()

original_df["variant"] = "original"
original_df["persona_text"] = (
    original_df[
        "original_persona"
    ].map(format_persona)
)

original_df["expected_label"] = (
    "NO_CONTRADICTION"
)

original_df["num_changed_facts"] = 0
original_df["num_grounded_changed_facts"] = 0

original_df["expected_label_binary"] = 0


augmented_df = selected_pairs_df.copy()

augmented_df["variant"] = "augmented"
augmented_df["persona_text"] = (
    augmented_df[
        "augmented_persona"
    ].map(format_persona)
)

augmented_df["expected_label"] = (
    "CONTRADICTION"
)

augmented_df["num_changed_facts"] = (
    augmented_df[
        "pair_num_augmented_facts"
    ]
)

augmented_df[
    "num_grounded_changed_facts"
] = (
    augmented_df[
        "pair_num_grounded_augmented_facts"
    ]
)

augmented_df["expected_label_binary"] = 1

In [26]:
pair_ids = [
    f"pair_{i:04d}"
    for i in range(
        len(selected_pairs_df)
    )
]

original_df["pair_id"] = pair_ids
augmented_df["pair_id"] = pair_ids

In [27]:
experiment_df = pd.concat(
    [
        original_df,
        augmented_df,
    ],
    ignore_index=True,
)

experiment_df["example_id"] = (
    experiment_df["pair_id"]
    + "_"
    + experiment_df["variant"]
)

experiment_df = experiment_df.rename(
    columns={
        "num_augmented_facts": (
            "pair_num_augmented_facts"
        ),
        "num_grounded_augmented_facts": (
            "pair_num_grounded_augmented_facts"
        ),
    }
)

experiment_df["num_changed_facts"] = (
    experiment_df["variant"]
    .eq("augmented")
    .astype(int)
    * experiment_df[
        "pair_num_augmented_facts"
    ]
)

experiment_df[
    "num_grounded_changed_facts"
] = (
    experiment_df["variant"]
    .eq("augmented")
    .astype(int)
    * experiment_df[
        "pair_num_grounded_augmented_facts"
    ]
)

In [28]:
experiment_df = (
    experiment_df
    .sample(
        frac=1,
        random_state=RANDOM_SEED,
    )
    .reset_index(drop=True)
)

In [29]:
assert (
    len(experiment_df)
    == 2 * len(selected_pairs_df)
)

assert experiment_df[
    "example_id"
].is_unique

assert (
    experiment_df
    .groupby("pair_id")
    .size()
    .eq(2)
    .all()
)

assert (
    experiment_df
    .groupby("pair_id")[
        "dialogue_text"
    ]
    .nunique()
    .eq(1)
    .all()
)

assert (
    experiment_df
    .groupby("pair_id")[
        "dialogue_id"
    ]
    .nunique()
    .eq(1)
    .all()
)

assert (
    experiment_df[
        "pair_id"
    ].nunique()
    == experiment_df[
        "dialogue_id"
    ].nunique()
)

print(
    "Experimental dataset validation passed."
)

Experimental dataset validation passed.


### 2.1 Input characteristics

Several simple textual characteristics are computed before LLM evaluation.
These variables are later used to investigate whether judge decisions or
inter-judge disagreement depend on properties of the input.

The following characteristics are considered:

- complete persona length,
- number of persona facts,
- dialogue length,
- total textual input length,
- lexical overlap between the complete persona and the dialogue.

For augmented profiles, the source data additionally provide the number of
changed persona facts and the number of changed facts grounded in the dialogue.

Lengths are measured in word tokens using a simple deterministic tokenizer.
Lexical overlap is measured using Jaccard similarity between the sets of words
appearing in the persona and dialogue.

In [30]:
WORD_PATTERN = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")


def tokenize_words(text: str) -> list[str]:
    """Return lowercase word tokens from English text."""
    return WORD_PATTERN.findall(str(text).lower())


def jaccard_overlap(text_a: str, text_b: str) -> float:
    """Compute Jaccard similarity between unique word sets."""
    words_a = set(tokenize_words(text_a))
    words_b = set(tokenize_words(text_b))

    union = words_a | words_b

    if not union:
        return 0.0

    return len(words_a & words_b) / len(union)

In [31]:
experiment_df[
    "persona_length"
] = (
    experiment_df[
        "persona_text"
    ]
    .map(tokenize_words)
    .map(len)
)

experiment_df[
    "dialogue_length"
] = (
    experiment_df[
        "dialogue_text"
    ]
    .map(tokenize_words)
    .map(len)
)

experiment_df[
    "input_length"
] = (
    experiment_df[
        "persona_length"
    ]
    + experiment_df[
        "dialogue_length"
    ]
)

experiment_df[
    "lexical_overlap"
] = experiment_df.apply(
    lambda row: jaccard_overlap(
        row["persona_text"],
        row["dialogue_text"],
    ),
    axis=1,
)

experiment_df[
    [
        "persona_fact_count",
        "persona_length",
        "dialogue_length",
        "input_length",
        "lexical_overlap",
        "pair_num_augmented_facts",
        "pair_num_grounded_augmented_facts",
        "num_changed_facts",
        "num_grounded_changed_facts",
    ]
].describe()

,persona_fact_count,persona_length,dialogue_length,input_length,lexical_overlap,pair_num_augmented_facts,pair_num_grounded_augmented_facts,num_changed_facts,num_grounded_changed_facts
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,4.540000,26.821500,165.115000,191.936500,0.141222,1.494000,1.491000,0.747000,0.745500
std,0.508458,6.706648,28.514247,29.638061,0.043630,0.687168,0.685678,0.891285,0.889452
min,3.000000,11.000000,74.000000,99.000000,0.025424,1.000000,1.000000,0.000000,0.000000
25%,4.000000,22.000000,146.000000,172.000000,0.111111,1.000000,1.000000,0.000000,0.000000
50%,5.000000,26.000000,164.000000,192.000000,0.135593,1.000000,1.000000,0.500000,0.500000
75%,5.000000,31.000000,184.000000,210.000000,0.165158,2.000000,2.000000,1.000000,1.000000
max,5.000000,67.000000,329.000000,352.000000,0.328947,4.000000,4.000000,4.000000,4.000000


In [32]:
experiment_summary = pd.Series(
    {
        "examples": (
            len(experiment_df)
        ),
        "pairs": (
            experiment_df[
                "pair_id"
            ].nunique()
        ),
        "dialogues": (
            experiment_df[
                "dialogue_id"
            ].nunique()
        ),
        "original_examples": (
            experiment_df[
                "variant"
            ].eq("original").sum()
        ),
        "augmented_examples": (
            experiment_df[
                "variant"
            ].eq("augmented").sum()
        ),
    },
    name="count",
)

experiment_summary.to_frame()

,count
examples,2000
pairs,1000
dialogues,1000
original_examples,1000
augmented_examples,1000


In [33]:
pd.crosstab(
    selected_pairs_df[
        "pair_num_augmented_facts"
    ],
    selected_pairs_df[
        "pair_num_grounded_augmented_facts"
    ],
    rownames=["augmented facts"],
    colnames=["grounded augmented facts"],
)

grounded augmented facts,1.0,2.0,3.0,4.0
augmented facts,,,,
1.0,606,0,0,0
2.0,2,303,0,0
3.0,0,1,77,0
4.0,0,0,0,11


In [34]:
pd.crosstab(
    selected_pairs_df["pair_num_augmented_facts"],
    selected_pairs_df["pair_num_grounded_augmented_facts"],
)

pair_num_grounded_augmented_facts,1.0,2.0,3.0,4.0
pair_num_augmented_facts,,,,
1.0,606,0,0,0
2.0,2,303,0,0
3.0,0,1,77,0
4.0,0,0,0,11


### 2.2 Final experimental sample

The resulting experimental sample is treated as fixed for all subsequent LLM
evaluation.

The sample contains matched original and augmented persona–dialogue examples.
Both members of a pair contain exactly the same dialogue. The original variant
contains the complete original persona, while the augmented variant contains
the corresponding contradiction-oriented persona produced by the rule-based
augmentation pipeline.

The dataset is generated deterministically using a fixed random seed. No
examples are added, removed, or resampled based on the outputs of the LLM
judges.

In [35]:
EXPERIMENT_DATA_PATH = (
    DATA_DIR
    / "llm_judge_experiment_v2.parquet"
)

experiment_df.to_parquet(
    EXPERIMENT_DATA_PATH,
    index=False,
)

print(
    f"Saved {len(experiment_df):,} examples to "
    f"{EXPERIMENT_DATA_PATH}"
)

Saved 2,000 examples to data/llm_judge_experiment_v2.parquet


### 2.3 Frozen pilot sample


In [36]:
PILOT_PAIRS = 50

pilot_pair_ids = (
    experiment_df[
        ["pair_id"]
    ]
    .drop_duplicates()
    .sample(
        n=PILOT_PAIRS,
        random_state=RANDOM_SEED,
    )
    ["pair_id"]
)

pilot_df = (
    experiment_df[
        experiment_df[
            "pair_id"
        ].isin(pilot_pair_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(pilot_df) == 2 * PILOT_PAIRS

print(
    f"Pilot: {PILOT_PAIRS} pairs / "
    f"{len(pilot_df)} examples"
)

Pilot: 50 pairs / 100 examples



Before the full LLM evaluation, a fixed pilot sample of 50 matched pairs
is used to verify the prompt, inference pipeline, output parsing, and judge
behaviour.

The same frozen pilot examples are evaluated by all four judges.

In [37]:
PILOT_DATA_PATH = (
    DATA_DIR
    / "llm_judge_pilot_v1.parquet"
)

pilot_df.to_parquet(
    PILOT_DATA_PATH,
    index=False,
)

print(
    f"Saved {len(pilot_df)} pilot examples "
    f"({pilot_df['pair_id'].nunique()} pairs) "
    f"to {PILOT_DATA_PATH}"
)

Saved 100 pilot examples (50 pairs) to data/llm_judge_pilot_v1.parquet


## 3. LLM judges

The same experimental examples are evaluated independently by several
instruction-tuned large language models.

Each judge receives:

- the complete persona of the speaker labelled `SELF`,
- the corresponding dialogue,
- the same contradiction-detection instructions.

The model must return a binary decision:

- `CONTRADICTION`
- `NO_CONTRADICTION`

The primary analysis compares decisions between judges rather than treating
the automatically generated expected labels as manually verified ground truth.

### 3.1 Evaluation prompt

All judges use the same prompt and binary decision rule.

A contradiction is present when something stated or clearly implied by `SELF`
is incompatible with at least one statement in the complete persona.

The prompt explicitly distinguishes:

- `SELF` from `PARTNER`,
- contradiction from missing evidence,
- incompatible information from additional compatible information,
- temporal and modal differences from genuine contradictions.

Judges are instructed to use only information contained in the persona and
dialogue and to return one of two labels:

- `CONTRADICTION`
- `NO_CONTRADICTION`

The exact prompt and output parser are defined in `judge_config.py`.
The final prompt used in the experiment is identified by `full_persona_v3`.

In [38]:
example = experiment_df[
    experiment_df[
        "variant"
    ].eq("original")
].iloc[0]

print(
    build_judge_prompt(
        persona=example[
            "persona_text"
        ],
        dialogue=example[
            "dialogue_text"
        ],
    )
)

You are a binary judge of persona consistency.

Determine whether statements made by SELF in the dialogue contradict at least one statement in SELF's persona.

Rules:
- CONTRADICTION: SELF explicitly states or clearly implies information that cannot be true at the same time as at least one persona statement.
- NO_CONTRADICTION: no such conflict is present.
- One conflicting persona statement is sufficient for CONTRADICTION.
- A persona statement that is not discussed in the dialogue is not a contradiction.
- Claims made only by PARTNER are not evidence about SELF. Use PARTNER's utterances only as conversational context.
- Respect temporal and modal information. Past and present statements, intentions, wishes, and current facts are contradictions only when they are genuinely incompatible.
- Additional compatible information is not a contradiction.
- Use only the information in the persona and dialogue. Do not assume unstated facts.

Persona of SELF:
- i need a wheel chair to get around.

In [39]:
example = experiment_df[
    experiment_df[
        "variant"
    ].eq("augmented")
].iloc[0]

print(
    build_judge_prompt(
        persona=example[
            "persona_text"
        ],
        dialogue=example[
            "dialogue_text"
        ],
    )
)

You are a binary judge of persona consistency.

Determine whether statements made by SELF in the dialogue contradict at least one statement in SELF's persona.

Rules:
- CONTRADICTION: SELF explicitly states or clearly implies information that cannot be true at the same time as at least one persona statement.
- NO_CONTRADICTION: no such conflict is present.
- One conflicting persona statement is sufficient for CONTRADICTION.
- A persona statement that is not discussed in the dialogue is not a contradiction.
- Claims made only by PARTNER are not evidence about SELF. Use PARTNER's utterances only as conversational context.
- Respect temporal and modal information. Past and present statements, intentions, wishes, and current facts are contradictions only when they are genuinely incompatible.
- Additional compatible information is not a contradiction.
- Use only the information in the persona and dialogue. Do not assume unstated facts.

Persona of SELF:
- i do not like fantasizing.
- i like

### 3.2 Pilot judge analysis

Before running the full experiment, each LLM judge is evaluated on the same
pilot sample of matched original and augmented persona–dialogue pairs.

For each judge, the analysis reports:

- output validity,
- distribution of predicted labels,
- predictions separately for original and augmented variants,
- agreement with the automatically derived expected labels,
- matched-pair prediction transitions,
- contradiction rates,
- examples resembling false negatives and false positives.

The expected labels are used as an experimental reference rather than manually
verified ground truth.

In [40]:
def load_judge_results(
    path: str | Path,
    expected_prompt_id: str = PROMPT_ID,
) -> pd.DataFrame:
    df = pd.read_parquet(path).copy()

    prompt_ids = df[
        "prompt_id"
    ].dropna().unique()

    if (
        len(prompt_ids) != 1
        or prompt_ids[0]
        != expected_prompt_id
    ):
        raise ValueError(
            f"{path}: expected prompt "
            f"{expected_prompt_id!r}, "
            f"found {prompt_ids}."
        )

    parsed = df[
        "raw_output"
    ].map(parse_judge_output)

    df["parsed_label"] = parsed.map(
        lambda result: result[0]
    )

    df["parsed_label_binary"] = parsed.map(
        lambda result: result[1]
    )

    df["valid_output"] = (
        df["parsed_label"].notna()
    )

    return df

def analyze_judge_results(
    df: pd.DataFrame,
    model_label: str,
) -> dict:
    print("=" * 70)
    print(model_label)
    print("=" * 70)

    print(f"Rows: {len(df):,}")
    print(
        f"Pairs: "
        f"{df['pair_id'].nunique():,}"
    )
    print(
        "Valid output rate:",
        f"{df['valid_output'].mean():.1%}",
    )

    print("\nParsed labels:")
    label_counts = (
        df["parsed_label"]
        .value_counts(dropna=False)
    )
    print(label_counts)

    print("\nPredictions by variant:")
    variant_predictions = pd.crosstab(
        df["variant"],
        df["parsed_label"],
        dropna=False,
    )
    display(variant_predictions)

    valid = df[
        df["valid_output"]
    ].copy()

    valid["matches_expected"] = (
        valid["parsed_label_binary"]
        == valid["expected_label_binary"]
    )

    expected_agreement = (
        valid["matches_expected"].mean()
    )

    print(
        "\nAgreement with expected labels:",
        f"{expected_agreement:.1%}",
    )

    agreement_by_variant = (
        valid
        .groupby("variant")[
            "matches_expected"
        ]
        .mean()
    )

    print(
        "\nAgreement with expected labels "
        "by variant:"
    )
    print(
        agreement_by_variant.map(
            lambda value: f"{value:.1%}"
        )
    )

    contradiction_rate = (
        valid[
            "parsed_label_binary"
        ].mean()
    )

    print(
        "\nOverall contradiction rate:",
        f"{contradiction_rate:.1%}",
    )

    contradiction_by_variant = (
        valid
        .groupby("variant")[
            "parsed_label_binary"
        ]
        .mean()
    )

    print(
        "\nContradiction rate by variant:"
    )
    print(
        contradiction_by_variant.map(
            lambda value: f"{value:.1%}"
        )
    )

    pair_predictions = (
        valid
        .pivot(
            index="pair_id",
            columns="variant",
            values="parsed_label",
        )
        .dropna(
            subset=[
                "original",
                "augmented",
            ]
        )
        .copy()
    )

    pair_predictions["transition"] = (
        pair_predictions["original"]
        + " → "
        + pair_predictions["augmented"]
    )

    transition_counts = (
        pair_predictions[
            "transition"
        ].value_counts()
    )

    print(
        "\nMatched-pair transitions:"
    )
    print(transition_counts)

    ideal_transition_rate = (
        pair_predictions[
            "transition"
        ]
        .eq(
            "NO_CONTRADICTION → CONTRADICTION"
        )
        .mean()
    )

    print(
        "\nExpected N → C transition rate:",
        f"{ideal_transition_rate:.1%}",
    )

    invalid_outputs = df[
        ~df["valid_output"]
    ][
        [
            "example_id",
            "variant",
            "raw_output",
        ]
    ].copy()

    if not invalid_outputs.empty:
        print(
            f"\nInvalid outputs: "
            f"{len(invalid_outputs)}"
        )
        display(invalid_outputs)

    return {
        "valid": valid,
        "label_counts": label_counts,
        "variant_predictions": variant_predictions,
        "agreement_by_variant": agreement_by_variant,
        "contradiction_by_variant": contradiction_by_variant,
        "pair_predictions": pair_predictions,
        "transition_counts": transition_counts,
        "invalid_outputs": invalid_outputs,
    }

def get_judge_error_examples(
    judgments_df: pd.DataFrame,
    experiment_df: pd.DataFrame,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
]:
    """
    Return augmented examples predicted as NO_CONTRADICTION and
    original examples predicted as CONTRADICTION.

    These resemble false negatives and false positives relative to the
    automatically derived expected labels.
    """
    inspection = judgments_df.merge(
        experiment_df[
            [
                "example_id",
                "persona_text",
                "dialogue_text",
                "changed_original_facts",
                "changed_augmented_facts",
                "changed_grounding_labels",
            ]
        ],
        on="example_id",
        how="left",
    )

    false_negative_like = inspection[
        inspection["variant"].eq(
            "augmented"
        )
        & inspection[
            "parsed_label"
        ].eq(
            "NO_CONTRADICTION"
        )
    ].copy()

    false_positive_like = inspection[
        inspection["variant"].eq(
            "original"
        )
        & inspection[
            "parsed_label"
        ].eq(
            "CONTRADICTION"
        )
    ].copy()

    return (
        false_negative_like,
        false_positive_like,
    )

In [41]:
JUDGE_CONFIGS = {
    "qwen3_4b": {
        "label": "Qwen3-4B",
        "file": "qwen3_4b_pilot_v3.parquet",
    },
    "phi4_mini": {
        "label": "Phi-4-mini-instruct",
        "file": "phi4_mini_pilot_v3.parquet",
    },
    "llama32_3b": {
        "label": "Llama-3.2-3B-Instruct",
        "file": "llama32_3b_pilot_v3.parquet",
    },
    "mistral7b": {
        "label": "Mistral-7B-Instruct-v0.3",
        "file": "mistral7b_pilot_v3.parquet",
    },
}

def run_pilot_analysis(
    model_id: str,
):
    config = JUDGE_CONFIGS[
        model_id
    ]

    judgments = load_judge_results(
        JUDGMENTS_DIR
        / config["file"]
    )

    analysis = analyze_judge_results(
        judgments,
        model_label=config["label"],
    )

    false_negative_like, false_positive_like = (
        get_judge_error_examples(
            judgments,
            experiment_df,
        )
    )

    print(
        "\nAugmented → NO_CONTRADICTION:",
        len(false_negative_like),
    )

    print(
        "Original → CONTRADICTION:",
        len(false_positive_like),
    )

    return (
        judgments,
        analysis,
        false_negative_like,
        false_positive_like,
    )

In [42]:
(
    qwen_pilot,
    qwen_analysis,
    qwen_false_negative_like,
    qwen_false_positive_like,
) = run_pilot_analysis(
    "qwen3_4b"
)

FileNotFoundError: [Errno 2] No such file or directory: 'data/outputs/qwen3_4b_pilot_v3.parquet'

In [ ]:
(
    phi_pilot,
    phi_analysis,
    phi_false_negative_like,
    phi_false_positive_like,
) = run_pilot_analysis(
    "phi4_mini"
)

In [ ]:
(
    llama_pilot,
    llama_analysis,
    llama_false_negative_like,
    llama_false_positive_like,
) = run_pilot_analysis(
    "llama32_3b"
)

In [ ]:
(
    mistral_pilot,
    mistral_analysis,
    mistral_false_negative_like,
    mistral_false_positive_like,
) = run_pilot_analysis(
    "mistral7b"
)

### 3.3 Inter-judge pilot comparison

After inspecting each judge individually, agreement is evaluated directly
between judges on the common pilot sample.

The pilot comparison is used to verify that all four judges produce usable
outputs and exhibit non-trivial decision behaviour before running the full
experiment.

The main pilot statistics are:

- raw pairwise agreement,
- Cohen's kappa,
- agreement separately for original and augmented variants,
- judge-specific contradiction rates.

In [ ]:
from sklearn.metrics import cohen_kappa_score


def compare_judges(
    judge_a: pd.DataFrame,
    judge_b: pd.DataFrame,
    label_a: str,
    label_b: str,
) -> pd.DataFrame:
    """
    Compare two judges on examples for which both produced valid labels.
    """
    comparison = (
        judge_a[
            [
                "example_id",
                "variant",
                "parsed_label",
                "parsed_label_binary",
            ]
        ]
        .rename(
            columns={
                "parsed_label": "label_a",
                "parsed_label_binary": "binary_a",
            }
        )
        .merge(
            judge_b[
                [
                    "example_id",
                    "parsed_label",
                    "parsed_label_binary",
                ]
            ].rename(
                columns={
                    "parsed_label": "label_b",
                    "parsed_label_binary": "binary_b",
                }
            ),
            on="example_id",
            how="inner",
        )
    )

    common_valid = comparison[
        comparison["binary_a"].notna()
        & comparison["binary_b"].notna()
    ].copy()

    raw_agreement = (
        common_valid["binary_a"]
        == common_valid["binary_b"]
    ).mean()

    kappa = cohen_kappa_score(
        common_valid["binary_a"],
        common_valid["binary_b"],
    )

    print("=" * 70)
    print(f"{label_a} vs {label_b}")
    print("=" * 70)

    print(
        f"Common valid examples: "
        f"{len(common_valid):,}"
    )

    print(
        f"Raw agreement: "
        f"{raw_agreement:.1%}"
    )

    print(
        f"Cohen's kappa: "
        f"{kappa:.3f}"
    )

    print("\nDecision table:")

    display(
        pd.crosstab(
            common_valid["label_a"],
            common_valid["label_b"],
            rownames=[label_a],
            colnames=[label_b],
        )
    )

    print("\nAgreement by variant:")

    agreement_by_variant = (
        common_valid
        .assign(
            agrees=lambda df:
                df["binary_a"]
                == df["binary_b"]
        )
        .groupby("variant")["agrees"]
        .mean()
    )

    print(
        agreement_by_variant.map(
            lambda value: f"{value:.1%}"
        )
    )

    disagreements = common_valid[
        common_valid["binary_a"]
        != common_valid["binary_b"]
    ].copy()

    print(
        f"\nDisagreements: "
        f"{len(disagreements):,}"
    )

    return common_valid

In [ ]:
qwen_phi_comparison = compare_judges(
    qwen_pilot,
    phi_pilot,
    label_a="Qwen3-4B",
    label_b="Phi-4-mini",
)

In [ ]:
qwen_phi_disagreements = (
    qwen_phi_comparison[
        qwen_phi_comparison["binary_a"]
        != qwen_phi_comparison["binary_b"]
    ]
)

qwen_phi_disagreements[
    [
        "example_id",
        "variant",
        "label_a",
        "label_b",
    ]
].head(20)